In [1]:
import pandas as pd
import json
import plotly.graph_objects as go
from pathlib import Path

# Label Verfication

You should document: a table or histogram showing the microbleed count distribution across 
subjects (e.g., how many subjects have 0, 1–5, 5–20, or 20+ microbleeds), and a histogram of 
individual microbleed sizes in mm³ (converting from voxel count using the voxel spacing from the 
NIfTI header). These statistics belong in the shared baseline section of their final report. 

In [2]:
base = Path.cwd().parent
path = base / "data" / "nnUNet_raw" / "Dataset001_VALDO" / "verification_stats" / "stats.json"
with open(path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# mextracting cmb counts
counts = [
    subj["microbleed_count"]
    for subj in data["statistics"].values()
]

# converting to categories
bins = {
    "0": 0,
    "1–5": 0,
    "6–20": 0,
    "20+": 0
}

for c in counts:
    if c == 0:
        bins["0"] += 1
    elif 1 <= c <= 5:
        bins["1–5"] += 1
    elif 6 <= c <= 20:
        bins["6–20"] += 1
    else:
        bins["20+"] += 1

In [8]:
fig = go.Figure([
    go.Bar(
        x=list(bins.keys()),
        y=list(bins.values())
    )
])

fig.update_layout(
    template="plotly_white",
    xaxis_title="Microbleed Count Range",
    yaxis_title="Number of Subjects"
)

fig.show()
fig.write_image("cmb_distr_across_subj.png", width=1600, height=900, scale=3)


In [9]:
volumes = []

for subj in data["statistics"].values():
    volumes.extend(subj["microbleed_vol_mm3"])

import plotly.express as px

fig = px.histogram(
    volumes,
    nbins=30,
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Microbleed Volume (mm³)",
    yaxis_title="Frequency",
    showlegend=False
)

fig.show()
fig.write_image("cmb_vol_distr.png", width=1600, height=600, scale=3)


In [5]:
df = pd.DataFrame(list(bins.items()), columns=["Range", "Subjects"])
print(df)

  Range  Subjects
0     0        22
1   1–5        40
2  6–20         7
3   20+         3
